# 2D FP32 Training Export v0.3

This notebook is orchestration-only for the directory-based 2D export pipeline.

Flow:
1. Select `input_dir` and `output_root`
2. Build `ExportConfig`
3. Run `export_directory(config)`
4. Preview output manifests


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
import ipywidgets as widgets
from ipyfilechooser import FileChooser


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preprocessing' / 'export_2d_training.py').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing preprocessing/export_2d_training.py')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.export_2d_training import ExportConfig, export_directory

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))


Using repo root: `/home/mitch/development/raccoon-ball`

In [9]:
DEFAULT_INPUT_DIR = REPO_ROOT / 'datasets'
DEFAULT_OUTPUT_ROOT = REPO_ROOT / 'preprocessing' / '03.training-export' / 'output'

BACKEND = 'torch-cuda'  # Change to 'cpu' for debug fallback
SKIP_EXISTING = True

# Smoke-test controls
SOURCE_FILES = None  # Example: [DEFAULT_INPUT_DIR / 'normal/example.wav', DEFAULT_INPUT_DIR / 'abnormal/example.wav']
MAX_FILES = None     # Example: 2

# Feature / export configuration
TARGET_SR = 16000
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = 'hann'
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0
LOG_POWER_REFERENCE = 1.0
EPSILON = 1e-8
ENERGY_BINS = 24
LOW_ENERGY_FLOOR = 0.0
MIN_ACTIVE_BANDS_PER_FRAME = 0
WINDOW_FRAMES = 64
STRIDE_FRAMES = 32


## Select Input and Output Directories
If no explicit selection is made, defaults are used.


In [3]:
def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, REPO_ROOT, Path.cwd()]:
        if candidate.exists():
            return candidate
    return Path.cwd()


input_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_INPUT_DIR)))
input_dir_chooser.title = '<b>Input directory (recursive .wav scan)</b>'
input_dir_chooser.show_only_dirs = True
input_dir_chooser.use_dir_icons = True
input_dir_chooser.select_default = True

output_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_OUTPUT_ROOT)))
output_dir_chooser.title = '<b>Output root directory</b>'
output_dir_chooser.show_only_dirs = True
output_dir_chooser.use_dir_icons = True
output_dir_chooser.select_default = True

display(
    widgets.HBox([
        widgets.VBox([input_dir_chooser]),
        widgets.VBox([output_dir_chooser]),
    ])
)


In [35]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()


input_dir = resolve_selected_dir(input_dir_chooser, DEFAULT_INPUT_DIR)
output_root = resolve_selected_dir(output_dir_chooser, DEFAULT_OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)

display(Markdown(f'- `input_dir`: `{input_dir}`\n- `output_root`: `{output_root}`'))


- `input_dir`: `/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal`
- `output_root`: `/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal`

In [36]:
wav_candidates = sorted(p for p in input_dir.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')
if MAX_FILES is None:
    selected_preview = wav_candidates
else:
    selected_preview = wav_candidates[:MAX_FILES]

print(f'Total .wav files discovered under input_dir: {len(wav_candidates)}')
print(f'Files selected for this run (after MAX_FILES filter): {len(selected_preview)}')
for path in selected_preview[:10]:
    print(path)

if not selected_preview and SOURCE_FILES is None:
    raise RuntimeError(
        f'No .wav files found under input_dir={input_dir}. '
        'Pick a directory that contains WAV files or set SOURCE_FILES explicitly.'
    )


Total .wav files discovered under input_dir: 456
Files selected for this run (after MAX_FILES filter): 456
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0000.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0001.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0002.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0003.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0004.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0005.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0006.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0007.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0008.wav
/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal/+6-pump-00-ab-0009.wav


In [37]:
config = ExportConfig(
    input_dir=input_dir,
    output_root=output_root,
    backend=BACKEND,
    source_files=SOURCE_FILES,
    max_files=MAX_FILES,
    skip_existing=SKIP_EXISTING,
    target_sr=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    window=WINDOW,
    num_bands=NUM_BANDS,
    fmin=FMIN,
    fmax=FMAX,
    log_power_reference=LOG_POWER_REFERENCE,
    epsilon=EPSILON,
    normalize_band_energy=True,
    normalization_mode='per_clip_minmax',
    energy_bins=ENERGY_BINS,
    low_energy_floor=LOW_ENERGY_FLOOR,
    min_active_bands_per_frame=MIN_ACTIVE_BANDS_PER_FRAME,
    window_frames=WINDOW_FRAMES,
    stride_frames=STRIDE_FRAMES,
)
config


ExportConfig(input_dir=PosixPath('/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal'), output_root=PosixPath('/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal'), backend='torch-cuda', source_files=None, max_files=None, skip_existing=True, target_sr=16000, mono=True, audio_offset_seconds=0.0, max_audio_seconds=None, n_fft=1024, hop_length=256, win_length=1024, window='hann', num_bands=96, fmin=50.0, fmax=8000.0, log_power_reference=1.0, epsilon=1e-08, normalize_band_energy=True, normalization_mode='per_clip_minmax', energy_bins=24, low_energy_floor=0.0, min_active_bands_per_frame=0, window_frames=64, stride_frames=32, manifest_prefix=None)

In [38]:
summary = export_directory(config)
summary


{'run_id': '20260319_155233',
 'run_started_utc': '2026-03-19T15:52:33.443442+00:00',
 'run_finished_utc': '2026-03-19T15:52:57.018063+00:00',
 'input_dir': '/home/mitch/development/raccoon-ball/datasets/+6db-pump-abnormal',
 'output_root': '/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal',
 'backend': 'torch-cuda',
 'device': 'cuda',
 'num_source_files_selected': 456,
 'num_file_rows': 456,
 'num_window_rows': 8208,
 'status_counts': {'exported': 456},
 'files_manifest_path': '/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal/manifests/20260319_155233_files.parquet',
 'windows_manifest_path': '/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal/manifests/20260319_155233_windows.parquet',
 'config_snapshot_path': '/home/mitch/development/raccoon-ball/training-data/+6db-pump-abnormal/manifests/20260319_155233_config.json'}

In [39]:
from pathlib import Path
import shutil

files_manifest_path = Path(summary['files_manifest_path'])
windows_manifest_path = Path(summary['windows_manifest_path'])
config_snapshot_path = Path(summary['config_snapshot_path'])

canonical_manifests_dir = DEFAULT_OUTPUT_ROOT / 'manifests'
canonical_manifests_dir.mkdir(parents=True, exist_ok=True)

canonical_files_manifest_path = canonical_manifests_dir / files_manifest_path.name
canonical_windows_manifest_path = canonical_manifests_dir / windows_manifest_path.name
canonical_config_snapshot_path = canonical_manifests_dir / config_snapshot_path.name

if files_manifest_path.resolve().parent != canonical_manifests_dir.resolve():
    shutil.copy2(files_manifest_path, canonical_files_manifest_path)
    shutil.copy2(windows_manifest_path, canonical_windows_manifest_path)
    shutil.copy2(config_snapshot_path, canonical_config_snapshot_path)
else:
    canonical_files_manifest_path = files_manifest_path
    canonical_windows_manifest_path = windows_manifest_path
    canonical_config_snapshot_path = config_snapshot_path

summary = dict(summary)
summary['canonical_files_manifest_path'] = str(canonical_files_manifest_path)
summary['canonical_windows_manifest_path'] = str(canonical_windows_manifest_path)
summary['canonical_config_snapshot_path'] = str(canonical_config_snapshot_path)

files_df = pd.read_parquet(files_manifest_path)
windows_df = pd.read_parquet(windows_manifest_path)

display(Markdown('### Run Summary'))
display(pd.DataFrame([summary]))

display(Markdown('### Files Manifest (sample)'))
display(files_df.head(20))

display(Markdown('### Windows Manifest (sample)'))
display(windows_df.head(20))

display(Markdown(
    '### Canonical Manifests For Tensor Inspection\n'
    f'- `{canonical_windows_manifest_path}`\n'
    f'- `{canonical_files_manifest_path}`\n'
    f'- `{canonical_config_snapshot_path}`'
))



### Run Summary

,run_id,run_started_utc,run_finished_utc,input_dir,output_root,backend,device,num_source_files_selected,num_file_rows,num_window_rows,status_counts,files_manifest_path,windows_manifest_path,config_snapshot_path,canonical_files_manifest_path,canonical_windows_manifest_path,canonical_config_snapshot_path
0,20260319_155233,2026-03-19T15:52:33.443442+00:00,2026-03-19T15:52:57.018063+00:00,/home/mitch/development/raccoon-ball/datasets/...,/home/mitch/development/raccoon-ball/training-...,torch-cuda,cuda,456,456,8208,{'exported': 456},/home/mitch/development/raccoon-ball/training-...,/home/mitch/development/raccoon-ball/training-...,/home/mitch/development/raccoon-ball/training-...,/home/mitch/development/raccoon-ball/preproces...,/home/mitch/development/raccoon-ball/preproces...,/home/mitch/development/raccoon-ball/preproces...


### Files Manifest (sample)

,source_file,relative_source_path,tensor_npz_path,status,error,source_sample_rate,target_sample_rate,num_samples_target_sr,num_frames,num_windows,num_windows_total,audio_loader,backend,device,started_utc,finished_utc
0,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.443657+00:00,2026-03-19T15:52:33.498838+00:00
1,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0001.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.499143+00:00,2026-03-19T15:52:33.552230+00:00
2,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0002.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.552489+00:00,2026-03-19T15:52:33.603912+00:00
3,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0003.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.604164+00:00,2026-03-19T15:52:33.654410+00:00
4,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0004.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.654670+00:00,2026-03-19T15:52:33.704937+00:00
5,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0005.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.705191+00:00,2026-03-19T15:52:33.755370+00:00
6,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0006.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.755621+00:00,2026-03-19T15:52:33.808131+00:00
7,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0007.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.808424+00:00,2026-03-19T15:52:33.860814+00:00
8,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0008.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.861101+00:00,2026-03-19T15:52:33.911608+00:00
9,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0009.wav,/home/mitch/development/raccoon-ball/training-...,exported,None,16000,16000,160000,626,18,18,soundfile_fallback,torch-cuda,cuda,2026-03-19T15:52:33.911861+00:00,2026-03-19T15:52:33.961971+00:00


### Windows Manifest (sample)

,source_file,relative_source_path,tensor_npz_path,tensor_index,start_frame,end_frame_exclusive,start_sec,end_sec,window_index,frame_start,frame_end_exclusive
0,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,0,0,64,0.000,1.024,0,0,64
1,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,1,32,96,0.512,1.536,1,32,96
2,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,2,64,128,1.024,2.048,2,64,128
3,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,3,96,160,1.536,2.560,3,96,160
4,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,4,128,192,2.048,3.072,4,128,192
5,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,5,160,224,2.560,3.584,5,160,224
6,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,6,192,256,3.072,4.096,6,192,256
7,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,7,224,288,3.584,4.608,7,224,288
8,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,8,256,320,4.096,5.120,8,256,320
9,/home/mitch/development/raccoon-ball/datasets/...,+6-pump-00-ab-0000.wav,/home/mitch/development/raccoon-ball/training-...,9,288,352,4.608,5.632,9,288,352


### Canonical Manifests For Tensor Inspection
- `/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/manifests/20260319_155233_windows.parquet`
- `/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/manifests/20260319_155233_files.parquet`
- `/home/mitch/development/raccoon-ball/preprocessing/03.training-export/output/manifests/20260319_155233_config.json`

## Smoke-Test Tips
- Set `MAX_FILES = 2` to run a quick dry export.
- Or set `SOURCE_FILES = [Path(...), Path(...)]` for targeted normal/abnormal checks.
- Keep `SKIP_EXISTING = True` for restart-safe reruns.
